In [ ]:
#Start in Terminal by running this code:
conda create -n crispresso2 -c bioconda -c conda-forge crispresso2

conda activate crispresso2

CRISPResso --help
CRISPRessoBatch --help

python -m ipykernel install --user --name crispresso2 --display-name "CRISPResso2 env"

conda activate crispresso2

conda install -c conda-forge ipykernel
##conda activate crispresso2

# Install Jupyter Notebook (and/or Lab)
conda install -c conda-forge notebook jupyterlab
# or with mamba if you use it:
# mamba install -c conda-forge notebook jupyterlab

jupyter notebook


In [3]:
#Generate required folders within 'crispresso_project'

from pathlib import Path

for d in ["fastq", "refs", "results"]:
    Path(d).mkdir(exist_ok=True)

print("Subfolders:", [p.name for p in Path(".").iterdir() if p.is_dir()])


import os
print(os.getcwd())

Subfolders: ['results', 'refs', 'fastq']
/Users/pdevant/crispresso_project


In [1]:

#Make project folder and check that CRISPResso is available in this kernel

import os, subprocess, textwrap

# e.g. "/Users/pdevant/projects/crispresso"
project_dir = "/Users/pdevant/crispresso_project"

os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

print("Current directory:", os.getcwd())
print("Contents:", os.listdir("."))

# Check that CRISPResso is available in this kernel
result = subprocess.run(["CRISPResso", "--help"], capture_output=True, text=True)
print("CRISPResso help (first 20 lines):")
print("\n".join(result.stdout.splitlines()[:20]))


Current directory: /Users/pdevant/crispresso_project
Contents: []
CRISPResso help (first 20 lines):
usage: CRISPResso [-h] [--version] [-r1 FASTQ_R1] [-r2 FASTQ_R2]
                  [-a AMPLICON_SEQ] [-an AMPLICON_NAME]
                  [-amas AMPLICON_MIN_ALIGNMENT_SCORE]
                  [--default_min_aln_score DEFAULT_MIN_ALN_SCORE]
                  [--expand_ambiguous_alignments]
                  [--assign_ambiguous_alignments_to_first_reference]
                  [-g GUIDE_SEQ] [-gn GUIDE_NAME] [-fg FLEXIGUIDE_SEQ]
                  [-fh FLEXIGUIDE_HOMOLOGY] [-fgn FLEXIGUIDE_NAME]
                  [--flexiguide_gap_open_penalty FLEXIGUIDE_GAP_OPEN_PENALTY]
                  [--flexiguide_gap_extend_penalty FLEXIGUIDE_GAP_EXTEND_PENALTY]
                  [--discard_guide_positions_overhanging_amplicon_edge]
                  [-e EXPECTED_HDR_AMPLICON_SEQ] [-c CODING_SEQ]
                  [--config_file CONFIG_FILE] [-q MIN_AVERAGE_READ_QUALITY]
                  [-s MIN_SI

In [6]:
#generate Amplicon file


amplicons_txt = """\
amplicon_name\tamplicon_sequence\tsgRNA_sequence
TNFAIP3_g1\tAATGTGAAACGCCCAACTGCCCCTTCTTCATGTCTGTGAACACCCAGCCTTTATGCCATGAGTGCTCAGAGAGGCGGCAAAAGAATCAAAACAAACTCCCAAAGCTGAACTCCAAGCCGGGCCCTGAGGGGCTCCCTGGCATGGCGCTCGGGGCCTCTCGGGGAGAAGCCTATGAGCCCT\tTATGCCATGAGTGCTCAGAG
TNFAIP3_g2\tAATGCTAAGAAGTTTGGAATCAGGTTCCAATTTCGCCCCTTTGAAAGTGGGTGGAATTTACTTGCCTCTCCACTGGCCTGCCCAGGAATGCTACAGATACCCCATTGTTCTCGGCTATGACAGCCATCATTTTGTACCCTTGGTGACCCTGAAGGACAGTGGGCCTGGTGAGAAAACTGCATTAATTCAC\tCTGTCCTTCAGGGTCACCAA
ZFP36_g1\tTGGCAAGCTCTAGTTCCCTGCAGCTGGGGTGGGGCGTCGCCCTGCATTTTCAGGTGCCTTAACCGACCCATTTCCGCAGAGCCTCCTGTCGCTGAGCCCTGACGTGCCCGTGCCATCCGACCATGGAGGGACTGAGTCCAGCCCAGGCTGGGGCTCCTCGGGACCCTGGAGCCTGAGCCCCTCCGACTCCAGCCCGTCTGGGGTCACCTCCCGCCTGCCTGGCCGCTCCACCAGCCTAGTG\tCCCGTGCCATCCGACCATGG
ZFP36_g2\tGAGCTATGTCGGACCTTCTCAGAGAGTGGGCGCTGCCGCTACGGGGCCAAGTGCCAGTTTGCCCATGGCCTGGGCGAGCTGCGCCAGGCCAATCGCCACCCCAAATACAAGACGGAACTCTGTCACAAGTTCTACCTCCAGGGCCGCTGCCCCTACGGCTCTCGCTGCCACTTCATCCACAACCCTAGCGAAGACCTGGCGGCCCCGGGCCACCCTCCTGTGCTTCGCCAGAGCATCAGCTTCT\tAGAGTTCCGTCTTGTATTTG
DUSP1_g1\tTCCCAGGAGGATACGAAGCGTTTTCGGCTTCCTGCCCGGAGCTGTGCAGCAAACAGTCGACCCCCATGGGGCTCAGCCTTCCCCTGAGTACTAGCGTCCCTGACAGCGCGGAATCTGGGTGCAGTTCCTGCAGTACCCCACTCTACGATCAGGTTAGTAGGTGTCCCTGCCACAGGGAAGAAGTAAGAACTGGCAAAGGCATGGAAGAGTAGTGCCA\tACTAACCTGATCGTAGAGTG
DUSP1_g2\tGTATGAAGGTGCCGTGTCCCGGTGTCACTAATGGGGaaaaaaaaTGTCTTTGTATTCCCAGGAGGATACGAAGCGTTTTCGGCTTCCTGCCCGGAGCTGTGCAGCAAACAGTCGACCCCCATGGGGCTCAGCCTTCCCCTGAGTACTAGCGTCCCTGACAGCGCGGAATCTGGGTGCAGTTCCTGCAGTACCCCACTCTACGATCAGGTTAGT\tGCAAACAGTCGACCCCCATG
MITF_g1\tCCAAGTCCTGAGCTTGCCATGTCCAAACCAGCCTGGCGATCATGTCATGCCACCGGTGCCGGGGAGCAGCGCACCCAACAGCCCCATGGCTATGCTTACGCTTAACTCCAACTGTGAAAAAGAGGTAATTCATGTCTCCTCTCCTCTCCTGTTTTCTTATGCTAAATAACTTGTCTGTTGAATATGATTCCCACACTTAACTGTGGACTCTAGTTTGTCCTTGTGGCCA\tTAAGCGTAAGCATAGCCATG
MITF_g2\tGGCCTGGTCAGTTTCATGTTTGTGCCTGAAGGAAGAGCCGTCTGAAACTCACAAATAACAGCGCTGTTTTCTTTTCCCTCCATGGCTATGTTCAGGTGCAGACCCACCTCGAAAACCCCACCAAGTACCACATACAGCAAGCCCAACGGCAGCAGGTAAAGCAGTACCTTTCTACCACTTTAGCAAATAAACATGCCAACCAAGTCCTGA\tACCACATACAGCAAGCCCAA
FOS_g1\tCAGACTACGAGGCGTCATCCTCCCGCTGCAGCAGCGCGTCCCCGGCCGGGGATAGCCTCTCTTACTACCACTCACCCGCAGACTCCTTCTCCAGCATGGGCTCGCCTGTCAACGCGCAGGTAAGGCTGGCTTCCCGTCGCCGCGGGGCCGGGGGCTTGGGGTCGCGGAGGAGGAGACACCGGGCGGGACGCTCCAGTAGATGAGTAGGGGGCTCCC\tGTAGTAAGAGAGGCTATCCC
FOS_g2\tAGACACTTTTACTGAATGTCGGTCTTTTTTTGTGATTATTCTAGTTATCTCCAGAAGAAGAAGAGAAAAGGAGAATCCGAAGGGAAAGGAATAAGATGGCTGCAGCCAAATGCCGCAACCGGAGGAGGGAGCTGACTGATACACTCCAAGCGGTAGGTACTCTGTGGGTTGCTCCTTTTTAAAACTTAAGGGGAAAGTTGGAGATTGAGCATAAGGGCC\tGCTGACTGATACACTCCAAG
TNF_g1\tGGATCATCTTCTCGAACCCCGAGTGACAAGCCTGTAGCCCATGTTGTAGGTAAGAGCTCTGAGGATGTGTCTTGGAACTTGGAGGGCTAGGATTTGGGGATTGAAGCCCGGCTGATGGTAGGCAGAACTTGGAGACAATGTGAGAAGGACTCGCTGAGCTCAAGGGAAGGGTGGAGGAACAGCACAGGCCTTAGTGGGATA\tAGAGCTCTTACCTACAACAT
TNF_g2\tGCAGCTCCAGTGGCTGAACCGCCGGGCCAATGCCCTCCTGGCCAATGGCGTGGAGCTGAGAGATAACCAGCTGGTGGTGCCATCAGAGGGCCTGTACCTCATCTACTCCCAGGTCCTCTTCAAGGGCCAAGGCTGCCCCTCCACCCATGTGCTCCTCACCCACACCATCAGCCGCATCGCCGTCTCCTACCAGACCAAGGTCAACCTC\tGGAGCTGAGAGATAACCAGC
TLR4_g1\tCTCTAGAGGGCCTGTGCAATTTGACCATTGAAGAATTCCGATTAGCATACTTAGACTACTACCTCGATGATATTATTGACTTATTTAATTGTTTGACAAATGTTTCTTCATTTTCCCTGGTGAGTGTGACTATTGAAAGGGTAAAAGACTTTTCTTATAATTTCGGATGGCAACATTTAGAATTAGTTAACTGTAAATTTGGACAGTTTCCCACATTGAAACTCAAATCTCTC\tATAAGTCAATAATATCATCG
TLR4_g2\tTCTAGAGCACTTGGACCTTTCCAGCAACAAGATTCAAAGTATTTATTGCACAGACTTGCGGGTTCTACATCAAATGCCCCTACTCAATCTCTCTTTAGACCTGTCCCTGAACCCTATGAACTTTATCCAACCAGGTGCATTTAAAGAAATTAGGCTTCATAAGCTGACTTTAAGAAATAATTTTGATAGTTTAAATGTAATGAAAACTTGTATTCAAGGTCTGGCTGGTT\tCCTATGAACTTTATCCAACC
TICAM2_g1\tATGAAGCCCTCAGAGTCCAGAATCTGCTACAAGATGACTTTGGTATCAAACCCGGAATAATCTTTGCTGAGATGCCATGTGGCAGACAGCATTTACAGAATTTAGATGATGCTGTAAATGGGTCTGCATGGACAATCTTATTACTGACTGAAAACTTTTTAAGAGATACTTGGTGTAATTTCCAGTTCTATACGTCCCTAATGAACTCCGTTAACAGGCAGC\tTGTAAATGCTGTCTGCCACA
TICAM2_g2\tTAATGGGTATCGGGAAGTCTAAAATAAATTCCTGCCCTCTTTCTCTCTCTTGGGGTAAAAGGCACAGTGTGGATACAAGTCCAGGATATCATGAGTCAGATTCCAAGAAGTCTGAAGATCTATCCTTGTGTAATGTTGCTGAGCACAGCAATACAACAGAGGGGCCAACAGGAAAGCAGGAGGGAGCTCAGAGCGTGGAAGAGATGTTTGAAG\tTTGGGGTAAAAGGCACAGTG
CD14_g1\tGGAACTGACGCTCGAGGACCTAAAGATAACCGGCACCATGCCTCCGCTGCCTCTGGAAGCCACAGGACTTGCACTTTCCAGCTTGCGCCTACGCAACGTGTCGTGGGCGACAGGGCGTTCTTGGCTCGCCGAGCTGCAGCAGTGGCTCAAGCCAGGCCTCAAGGTACTGAGCATTGCCCAAGCACACTCGCCTGCCTTTTCCTGCGAACAGGTTCGCGCCTT\tTCGCCCACGACACGTTGCGT
CD14_g2\tGACACGGTCAAGGCTCTCCGCGTGCGGCGGCTCACAGTGGGAGCCGCACAGGTTCCTGCTCAGCTACTGGTAGGCGCCCTGCGTGTGCTAGCGTACTCCCGCCTCAAGGAACTGACGCTCGAGGACCTAAAGATAACCGGCACCATGCCTCCGCTGCCTCTGGAAGCCACAGGACTTGCACTTTCCAGCTTGCGCCTACG\tGGAGTACGCTAGCACACGCA
MAP3K8_g1\tGCCAGCTTGACTGAAGTTCTCACCGTCTCACATTTTTATTTTGTTGTAGGTCATCACTCCCCAAAATGGACGTTACCAAATAGATTCCGATGTTCTCCTGATCCCCTGGAAGCTGACTTACAGGAATATTGGTTCTGATTTTATTCCTCGGGGCGCCTTTGGAAAGGTATACTTGGCACAAGATATAAAGACGAAGAAAAGAATGGCGTGTAAACTGGTATGTGT\tCCAGGGGATCAGGAGAACAT
MAP3K8_g2\tGAGTACATGAGCACTGGAAGTGACAATAAAGAAGAGATTGATTTATTAATTAAACATTTAAATGTGTCTGATGTAATAGACATTATGGAAAATCTTTATGCAAGTGAAGAGCCAGCAGTTTATGAACCCAGTCTAATGACCATGTGTCAAGACAGTAATCAAAACGATGAGCGTTCTAAGTCTCTGCTGCTTAGTGGCCAAGAGGTACCATGGTTGTCATCAGTCAGATATGGAACTGTGGAGGATTTGCTTGC\tTGACACATGGTCATTAGACT
B2M_ABE\tTGTCTTTCAGCAAGGACTGGTCTTTCTATCTCTTGTACTACACTGAATTCACCCCCACTGAAAAAGATGAGTATGCCTGCCGTGTGAACCATGTGACTTTGTCACAGCCCAAGATAGTTAAGTGGGGTAAGTCTTACATTCTTTTGTAAGCTGCTGAAAGTTGTGTATGAGTAGTCATATCATAAAGCTGCTTTGATATAAAAAAGGTCTATGGCCATACTACCCTGAATGAGTCCC\tcttaccccacttaactatct
"""


with open("refs/amplicons.txt", "w") as f:
    f.write(amplicons_txt)

print(open("refs/amplicons.txt").read())

amplicon_name	amplicon_sequence	sgRNA_sequence
TNFAIP3_g1	AATGTGAAACGCCCAACTGCCCCTTCTTCATGTCTGTGAACACCCAGCCTTTATGCCATGAGTGCTCAGAGAGGCGGCAAAAGAATCAAAACAAACTCCCAAAGCTGAACTCCAAGCCGGGCCCTGAGGGGCTCCCTGGCATGGCGCTCGGGGCCTCTCGGGGAGAAGCCTATGAGCCCT	TATGCCATGAGTGCTCAGAG
TNFAIP3_g2	AATGCTAAGAAGTTTGGAATCAGGTTCCAATTTCGCCCCTTTGAAAGTGGGTGGAATTTACTTGCCTCTCCACTGGCCTGCCCAGGAATGCTACAGATACCCCATTGTTCTCGGCTATGACAGCCATCATTTTGTACCCTTGGTGACCCTGAAGGACAGTGGGCCTGGTGAGAAAACTGCATTAATTCAC	CTGTCCTTCAGGGTCACCAA
ZFP36_g1	TGGCAAGCTCTAGTTCCCTGCAGCTGGGGTGGGGCGTCGCCCTGCATTTTCAGGTGCCTTAACCGACCCATTTCCGCAGAGCCTCCTGTCGCTGAGCCCTGACGTGCCCGTGCCATCCGACCATGGAGGGACTGAGTCCAGCCCAGGCTGGGGCTCCTCGGGACCCTGGAGCCTGAGCCCCTCCGACTCCAGCCCGTCTGGGGTCACCTCCCGCCTGCCTGGCCGCTCCACCAGCCTAGTG	CCCGTGCCATCCGACCATGG
ZFP36_g2	GAGCTATGTCGGACCTTCTCAGAGAGTGGGCGCTGCCGCTACGGGGCCAAGTGCCAGTTTGCCCATGGCCTGGGCGAGCTGCGCCAGGCCAATCGCCACCCCAAATACAAGACGGAACTCTGTCACAAGTTCTACCTCCAGGGCCGCTGCCCCTACGGCTCTCGCTGCCACTTCATCCACAACCCTAGCGAAGACCTGGCGGCCCCGGGCCACCCTCCTGTGCTTCGCCAGAGCAT

In [23]:
##Generate batch files

import pandas as pd
from pathlib import Path
import re

fastq_dir = Path("fastq")

# 1. Collect all FASTQ filenames
fastqs = sorted([f.name for f in fastq_dir.glob("*.fastq.gz")])
len(fastqs), fastqs[:5]


def extract_sample_id(filename):
    """
    Extract sample ID from filenames like:
      CD14_g1_D1_S141_L001_R1_001.fastq.gz
    → CD14_g1_D1
    """
    return filename.split("_S")[0]

# Test:
for f in fastqs[:10]:
    print(f, "→", extract_sample_id(f))



data = {}
for f in fastqs:
    sample_id = extract_sample_id(f)
    if sample_id not in data:
        data[sample_id] = {"sample_name": sample_id}
    if "_R1_" in f:
        data[sample_id]["fastq_r1"] = f"fastq/{f}"
    if "_R2_" in f:
        data[sample_id]["fastq_r2"] = f"fastq/{f}"

df = pd.DataFrame(data.values())
df = df.sort_values("sample_name")
df.head()


def is_abe(sample_name):
    return ("ABE" in sample_name) or ("CBE" in sample_name)

df["is_ABE"] = df["sample_name"].apply(is_abe)

df["abe_amplicon"] = df["is_ABE"].apply(lambda x: "B2M_ABE" if x else None)



# True for ABE/CBE/BE samples, False for pure KO / other controls
df["is_ABE"] = df["sample_name"].str.contains("ABE|CBE|BE_")
df[["sample_name", "is_ABE"]].head(20)


import numpy as np

# Start with all NaN
df["amplicon_name"] = np.nan

# KO samples: derive GENE_g1 / GENE_g2 from sample_name
mask_ko = ~df["is_ABE"]
df.loc[mask_ko, "amplicon_name"] = (
    df.loc[mask_ko, "sample_name"]
      .str.extract(r"^([^_]+_g[12])")[0]   # captures e.g. "TNFAIP3_g1"
)

# ABE samples: all share the same amplicon name
mask_abe = df["is_ABE"]
df.loc[mask_abe, "amplicon_name"] = "B2M_ABE"

df.head(20)


ko_df = df[~df["is_ABE"]].copy()

ko_df = ko_df[["sample_name", "fastq_r1", "fastq_r2", "amplicon_name"]]

ko_out = Path("refs/KO_batch.txt")
ko_df.to_csv(ko_out, sep="\t", index=False)
print(f"Wrote {ko_out}")
ko_df.head()



abe_df = df[df["is_ABE"]].copy()

abe_df = abe_df[["sample_name", "fastq_r1", "fastq_r2", "amplicon_name"]]

abe_out = Path("refs/ABE_batch.txt")
abe_df.to_csv(abe_out, sep="\t", index=False)
print(f"Wrote {abe_out}")
abe_df.head()


CD14_g1_D1_S141_L001_R1_001.fastq.gz → CD14_g1_D1
CD14_g1_D1_S141_L001_R2_001.fastq.gz → CD14_g1_D1
CD14_g1_D2_S142_L001_R1_001.fastq.gz → CD14_g1_D2
CD14_g1_D2_S142_L001_R2_001.fastq.gz → CD14_g1_D2
CD14_g1_D3_S143_L001_R1_001.fastq.gz → CD14_g1_D3
CD14_g1_D3_S143_L001_R2_001.fastq.gz → CD14_g1_D3
CD14_g1_D4_S144_L001_R1_001.fastq.gz → CD14_g1_D4
CD14_g1_D4_S144_L001_R2_001.fastq.gz → CD14_g1_D4
CD14_g1_D5_S145_L001_R1_001.fastq.gz → CD14_g1_D5
CD14_g1_D5_S145_L001_R2_001.fastq.gz → CD14_g1_D5
Wrote refs/KO_batch.txt
Wrote refs/ABE_batch.txt


/var/folders/9l/dyp50m2d6s30t17769kjdmt00000gr/T/ipykernel_68403/4023315626.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['CD14_g1' 'CD14_g1' 'CD14_g1' 'CD14_g1' 'CD14_g1' 'CD14_g1' 'CD14_g2'
 'CD14_g2' 'CD14_g2' 'CD14_g2' 'CD14_g2' 'CD14_g2' nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan 'DUSP1_g1' 'DUSP1_g1'
 'DUSP1_g1' 'DUSP1_g1' 'DUSP1_g1' 'DUSP1_g1' 'DUSP1_g2' 'DUSP1_g2'
 'DUSP1_g2' 'DUSP1_g2' 'DUSP1_g2' 'DUSP1_g2' 'FOS_g1' 'FOS_g1' 'FOS_g1'
 'FOS_g1' 'FOS_g1' 'FOS_g1' 'FOS_g2' 'FOS_g2' 'FOS_g2' 'FOS_g2' 'FOS_g2'
 'FOS_g2' 'MAP3K8_g1' 'MAP3K8_g1' 'MAP3K8_g1' 'MAP3K8_g1' 'MAP3K8_g1'
 'MAP3K8_g1' 'MAP3K8_g2' 'MAP3K8_g2' 'MAP3K8_g2' 'MAP3K8_g2' 'MAP3K8_g2'
 'MAP3K8_g2' 'MITF_g1' 'MITF_g1' 'MITF_g1' 'MITF_g1' 'MITF_g1' 'MITF_g1'
 'MITF_g2' 'MITF_g2' 'MITF_g2' 'MITF_g2' 'MITF_g2' 'MITF_g2' nan
 'TICAM2_g1' 'TICAM2_g1' 'TICAM2_g1' 'TICAM2_g1' 'TICAM2_g1' 'TICAM2_g1'
 'TIC

,sample_name,fastq_r1,fastq_r2,amplicon_name
19,D1_A14_NGG-ABE_0-2_ul,fastq/D1_A14_NGG-ABE_0-2_ul_S5_L001_R1_001.fas...,fastq/D1_A14_NGG-ABE_0-2_ul_S5_L001_R2_001.fas...,B2M_ABE
20,D1_A14_NGG-ABE_20_ul,fastq/D1_A14_NGG-ABE_20_ul_S7_L001_R1_001.fast...,fastq/D1_A14_NGG-ABE_20_ul_S7_L001_R2_001.fast...,B2M_ABE
21,D1_A14_NGG-ABE_2_ul,fastq/D1_A14_NGG-ABE_2_ul_S6_L001_R1_001.fastq.gz,fastq/D1_A14_NGG-ABE_2_ul_S6_L001_R2_001.fastq.gz,B2M_ABE
22,D1_B56_CBE_0-2_ul,fastq/D1_B56_CBE_0-2_ul_S2_L001_R1_001.fastq.gz,fastq/D1_B56_CBE_0-2_ul_S2_L001_R2_001.fastq.gz,B2M_ABE
23,D1_B56_CBE_20_ul,fastq/D1_B56_CBE_20_ul_S4_L001_R1_001.fastq.gz,fastq/D1_B56_CBE_20_ul_S4_L001_R2_001.fastq.gz,B2M_ABE


In [26]:
## Generate new batch file (merge of previous batch file and amplicon file)

import pandas as pd
from pathlib import Path

# Load your existing KO batch (sample_name, fastq_r1, fastq_r2, amplicon_name)
ko_batch = pd.read_csv("refs/KO_batch.txt", sep="\t")

# Load amplicon definitions (amplicon_name, amplicon_sequence, sgRNA_sequence)
amp = pd.read_csv("refs/amplicons.txt", sep="\t")

ko_batch.head(), amp.head()

merged = ko_batch.merge(amp, on="amplicon_name", how="left")

# Sanity check for missing mappings
missing = merged[merged["amplicon_sequence"].isna()]
print("Rows with missing amplicon_sequence:", len(missing))
if len(missing) > 0:
    display(missing[["sample_name", "amplicon_name"]])


ko_batch2 = merged.rename(columns={
    "sample_name": "name",
    "amplicon_sequence": "amplicon_seq",
    "sgRNA_sequence": "guide_seq",
})

ko_batch2 = ko_batch2[["name", "fastq_r1", "fastq_r2", "amplicon_seq", "guide_seq"]]

out_ko = Path("refs/KO_batch_CRISPResso2.txt")
ko_batch2.to_csv(out_ko, sep="\t", index=False)
print(f"Wrote {out_ko}")
ko_batch2.head()




Rows with missing amplicon_sequence: 20


,sample_name,amplicon_name
12,D1_A14_A90,NaN
13,D1_A14_B100,NaN
14,D1_A14_B96,NaN
15,D1_A14_B97,NaN
16,D1_A14_B98,NaN
17,D1_A14_B99,NaN
18,D1_A14_C01,NaN
19,D1_unreated,NaN
20,D2_A14_A90,NaN
21,D2_A14_B100,NaN


Wrote refs/KO_batch_CRISPResso2.txt


,name,fastq_r1,fastq_r2,amplicon_seq,guide_seq
0,CD14_g1_D1,fastq/CD14_g1_D1_S141_L001_R1_001.fastq.gz,fastq/CD14_g1_D1_S141_L001_R2_001.fastq.gz,GGAACTGACGCTCGAGGACCTAAAGATAACCGGCACCATGCCTCCG...,TCGCCCACGACACGTTGCGT
1,CD14_g1_D2,fastq/CD14_g1_D2_S142_L001_R1_001.fastq.gz,fastq/CD14_g1_D2_S142_L001_R2_001.fastq.gz,GGAACTGACGCTCGAGGACCTAAAGATAACCGGCACCATGCCTCCG...,TCGCCCACGACACGTTGCGT
2,CD14_g1_D3,fastq/CD14_g1_D3_S143_L001_R1_001.fastq.gz,fastq/CD14_g1_D3_S143_L001_R2_001.fastq.gz,GGAACTGACGCTCGAGGACCTAAAGATAACCGGCACCATGCCTCCG...,TCGCCCACGACACGTTGCGT
3,CD14_g1_D4,fastq/CD14_g1_D4_S144_L001_R1_001.fastq.gz,fastq/CD14_g1_D4_S144_L001_R2_001.fastq.gz,GGAACTGACGCTCGAGGACCTAAAGATAACCGGCACCATGCCTCCG...,TCGCCCACGACACGTTGCGT
4,CD14_g1_D5,fastq/CD14_g1_D5_S145_L001_R1_001.fastq.gz,fastq/CD14_g1_D5_S145_L001_R2_001.fastq.gz,GGAACTGACGCTCGAGGACCTAAAGATAACCGGCACCATGCCTCCG...,TCGCCCACGACACGTTGCGT


In [33]:
## Running CRISPResso with updated batch file for KO samples


!CRISPRessoBatch \
    --batch_settings refs/KO_batch_CRISPResso2.txt \
    --min_average_read_quality 25 \
    --min_single_bp_quality 20 \
    --plot_window_size 20 \
    --default_min_aln_score 40 \
    --name KO_run \
    --output_folder results/KO_batch
   


INFO  @ Thu, 11 Dec 2025 20:39:36 (0.0% done):
	 
                             ~~~CRISPRessoBatch~~~                              
       -Analysis of CRISPR/Cas9 outcomes from batch deep sequencing data-       
                                                                                
                _                                               _               
               '  )                                            '  )             
               .-'             _________________               .-'              
              (____           | __    ___ __    |             (____             
           C)|     \          ||__) /\ | /  |__||          C)|     \            
             \     /          ||__)/--\| \__|  ||            \     /            
              \___/           |_________________|             \___/             

                           [CRISPResso version 2.3.3]                           
[Note that as of version 2.3.0 FLASh and Trimmomatic have 

In [43]:
## Run CRISPResso on BE samples using ABE mode

!CRISPRessoBatch \
    --batch_settings refs/ABE-batch2.txt \
    --base_editor_output \
    --conversion_nuc_from A \
    --conversion_nuc_to G \
    --min_average_read_quality 25 \
    --min_single_bp_quality 20 \
    --default_min_aln_score 50 \
    --name BE_AtoG \
    --output_folder results/BE_AtoG

INFO  @ Fri, 12 Dec 2025 17:43:05 (0.0% done):
	 
                             ~~~CRISPRessoBatch~~~                              
       -Analysis of CRISPR/Cas9 outcomes from batch deep sequencing data-       
                                                                                
                _                                               _               
               '  )                                            '  )             
               .-'             _________________               .-'              
              (____           | __    ___ __    |             (____             
           C)|     \          ||__) /\ | /  |__||          C)|     \            
             \     /          ||__)/--\| \__|  ||            \     /            
              \___/           |_________________|             \___/             

                           [CRISPResso version 2.3.3]                           
[Note that as of version 2.3.0 FLASh and Trimmomatic have 

In [42]:
##RunCRISPResso in CBE mode

!CRISPRessoBatch \
    --batch_settings refs/ABE-batch2.txt \
    --base_editor_output \
    --base_edit \
    --conversion_nuc_from C \
    --conversion_nuc_to T \
    --min_average_read_quality 25 \
    --min_single_bp_quality 20 \
    --default_min_aln_score 50 \
    --name BE_CtoT \
    --output_folder results/BE_CtoT

INFO  @ Fri, 12 Dec 2025 17:26:58 (0.0% done):
	 
                             ~~~CRISPRessoBatch~~~                              
       -Analysis of CRISPR/Cas9 outcomes from batch deep sequencing data-       
                                                                                
                _                                               _               
               '  )                                            '  )             
               .-'             _________________               .-'              
              (____           | __    ___ __    |             (____             
           C)|     \          ||__) /\ | /  |__||          C)|     \            
             \     /          ||__)/--\| \__|  ||            \     /            
              \___/           |_________________|             \___/             

                           [CRISPResso version 2.3.3]                           
[Note that as of version 2.3.0 FLASh and Trimmomatic have 